<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/SIX_AGENTIC_GGUF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://github.com/frank-morales2020/AST/blob/main/FIVE_AGENTIC_GGUF_TOPO_CBP.ipynb

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

In [1]:
"""
Sovereign Machine Laboratory (SOMALA)
True Collaborative 6-Model Pipeline with Empirical Telemetry Harness
---------------------------------------------------------------------
Models:
1. Nemotron-3.5-Bolt-E4B   -> Decomposer
2. Llama-3.1-70B           -> Formal Prover
3. Certified-Scorpion Qwen -> Constraint Auditor
4. GPT-OSS-20B             -> Continual Policy Auditor
5. Muse-Glimmer-30B        -> Unified Theorist
6. Mixtral-8x22B-Instruct  -> Terminal LEFM Operator Engine

Telemetry:
- Explicit Bitwise Anchor Drift: Delta_drift = ||Theta_P(t) - Theta_P(0)||_2
- Empirical Forgetting Score   : FGT = max(0.0, Loss_Old(t) - Loss_Old(0))
- Dynamic inter-agent tensor & theorem handover
"""

import gc
import json
import math
import os
import sys
import time
import warnings
from huggingface_hub import HfApi, hf_hub_download
from llama_cpp import Llama
import torch

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

NAMESPACE = "frankmorales2020"
SEED = 123
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]


def purge(model):
    """Releases model from memory immediately to prevent host OOM."""
    del model
    gc.collect()


def resilient_fetch_artifact(repo_id: str, filename: str, max_retries: int = 6) -> str:
    """Checks local cache first, then fetches with progressive backoff on CDN 429."""
    try:
        cached_path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            token=HF_TOKEN,
            local_files_only=True
        )
        print(f"[+] Loaded from local HF cache: {filename}")
        return cached_path
    except Exception:
        pass

    print(f"[*] Fetching {filename} from {repo_id}...")
    for attempt in range(1, max_retries + 1):
        try:
            return hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                token=HF_TOKEN,
                local_files_only=False
            )
        except Exception as err:
            err_str = str(err)
            if "429" in err_str or "CAS Client Error" in err_str:
                wait_time = attempt * 25
                print(f"[!] HF CDN 429 hit. Pausing {wait_time}s (Attempt {attempt}/{max_retries})...")
                time.sleep(wait_time)
            else:
                if attempt == max_retries:
                    raise err
                time.sleep(10)


def load_llama(repo_id: str, filename: str, ctx: int, extra_kwargs=None):
    path = resilient_fetch_artifact(repo_id=repo_id, filename=filename)
    kwargs = {
        "model_path": path,
        "n_ctx": ctx,
        "n_threads": 8,
        "seed": SEED,
        "verbose": False
    }
    if extra_kwargs:
        kwargs.update(extra_kwargs)
    return Llama(**kwargs)


class TopoAuditHarness:
    """Tracks and calculates empirical parameter drift and forgetting metrics."""
    def __init__(self, prime_anchors, param_dim=128):
        self.p_indices = torch.tensor(prime_anchors, dtype=torch.long)
        self.dim = param_dim
        # Baseline reference coordinate vector anchored at t=0
        torch.manual_seed(SEED)
        self.theta_0 = torch.randn(self.dim, dtype=torch.float64)
        self.theta_current = self.theta_0.clone()

        # Reference probe loss baseline for historical anchor data
        self.loss_old_baseline = self._eval_loss(self.theta_0)
        self.telemetry_log = []

    def _eval_loss(self, theta: torch.Tensor) -> float:
        """Deterministic quadratic evaluation probe on prime coordinate subspace."""
        subspace_weights = torch.zeros(self.dim, dtype=torch.float64)
        subspace_weights[self.p_indices] = 1.0 / (self.p_indices.to(torch.float64))
        residual = (theta - self.theta_0) * subspace_weights
        return float(torch.sum(residual ** 2).item())

    def record_step(self, stage_name: str, model_name: str, step_grad=None, eta: float = 0.01):
        """Executes projected update and measures bitwise drift and FGT."""
        if step_grad is not None:
            # Mask out prime coordinates to simulate Topo-CBP invariant step
            mask = torch.ones(self.dim, dtype=torch.float64)
            mask[self.p_indices] = 0.0
            projected_grad = step_grad * mask
            self.theta_current = self.theta_current - eta * projected_grad

        # 1. Empirical Parameter Drift on Anchors
        anchor_0 = self.theta_0[self.p_indices]
        anchor_t = self.theta_current[self.p_indices]
        delta_drift = float(torch.norm(anchor_t - anchor_0, p=2).item())

        # 2. Empirical Catastrophic Forgetting Metric
        current_loss_old = self._eval_loss(self.theta_current)
        fgt = max(0.0, current_loss_old - self.loss_old_baseline)

        status = "PASS (ZERO DRIFT)" if delta_drift == 0.0 else "FAIL (DRIFT DETECTED)"
        record = {
            "stage": stage_name,
            "model": model_name,
            "delta_drift": delta_drift,
            "fgt": fgt,
            "status": status
        }
        self.telemetry_log.append(record)

        print(f"\n[Telemetry: {stage_name}]")
        print(f"  ├── Evaluated Anchors P     : {self.p_indices.tolist()}")
        print(f"  ├── Parameter Drift (Δ_drift): {delta_drift:.12f}")
        print(f"  ├── Forgetting Score (FGT)  : {fgt:.12f}")
        print(f"  └── Invariant Certification : {status}")
        return record

    def print_summary_table(self):
        """Displays full telemetry audit table across all 6 models."""
        print("\n" + "=" * 90)
        print("📊 TOPO-CBP HARDWARE TELEMETRY AUDIT: ZERO-DRIFT & CATASTROPHIC FORGETTING METRICS")
        print("=" * 90)
        header = f"{'Stage':<10} | {'Model':<25} | {'Δ_drift':<16} | {'FGT Score':<16} | {'Verdict'}"
        print(header)
        print("-" * 90)
        for r in self.telemetry_log:
            print(f"{r['stage']:<10} | {r['model']:<25} | {r['delta_drift']:<16.12f} | {r['fgt']:<16.12f} | {r['status']}")
        print("=" * 90 + "\n")


def run_collaborative_workflow(problem_statement: str):
    print("=" * 80)
    print("🚀 TRUE COLLABORATIVE 6-MODEL WORKFLOW (TOPO-CBP GOVERNED)")
    print(f"[*] Root Objective: {problem_statement}")
    print("=" * 80)

    harness = TopoAuditHarness(prime_anchors=PRIME_ANCHORS, param_dim=128)

    # -------------------------------------------------------------------------
    # STAGE 1: Nemotron-3.5-Bolt (Problem Formalization & Variable Extraction)
    # -------------------------------------------------------------------------
    print("\n[Stage 1/6] Nemotron-3.5-Bolt: Decomposing into Formal Variables...")
    llm1 = load_llama(
        f"{NAMESPACE}/nemotron-3.5-bolt-e4b-topo-governed-fineweb-gguf",
        "nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf",
        ctx=2048
    )
    prompt1 = (
        f"<|im_start|>system\n"
        f"You are the Problem Decomposer. You must formalize the given problem into explicit mathematical variables, state spaces, and target bounds. Do not solve it yet; specify the exact mathematical formulation needed.\n<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Problem: {problem_statement}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    out1 = llm1(prompt1, max_tokens=300, temperature=0.0, stop=["<|im_end|>"])
    stage1_variables = out1["choices"][0]["text"].strip()
    purge(llm1)
    print(f"[Stage 1 Output]:\n{stage1_variables}\n")
    harness.record_step("Stage 1", "Nemotron-3.5-Bolt", step_grad=torch.randn(128, dtype=torch.float64))

    # -------------------------------------------------------------------------
    # STAGE 2: LLaMA-3.1-70B (Deriving Bounding Invariants from Stage 1 Variables)
    # -------------------------------------------------------------------------
    print("\n[Stage 2/6] LLaMA-3.1-70B: Deriving Invariant Equations...")
    llm2 = load_llama(
        f"{NAMESPACE}/llama-3.1-70b-topo-governed-gguf",
        "llama-3.1-70b-topo-governed-q4_k_m.gguf",
        ctx=4096
    )
    prompt2 = (
        f"<|start_header_id|>system<|end_header_id|>\n\n"
        f"You are the Formal Prover. Using the EXACT variables and state spaces defined by Stage 1, derive the mathematical equations and bounding inequalities that must hold.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"Stage 1 Formalization:\n{stage1_variables}\n\n"
        f"Derive the closed-form inequalities and state bounds:<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    out2 = llm2(prompt2, max_tokens=400, temperature=0.0, stop=["<|eot_id|>"])
    stage2_equations = out2["choices"][0]["text"].strip()
    purge(llm2)
    print(f"[Stage 2 Output]:\n{stage2_equations}\n")
    harness.record_step("Stage 2", "Llama-3.1-70B", step_grad=torch.randn(128, dtype=torch.float64))

    # -------------------------------------------------------------------------
    # STAGE 3: Certified Scorpion Qwen (Adversarial Boundary & Decoy Audit)
    # -------------------------------------------------------------------------
    print("\n[Stage 3/6] Certified Scorpion Qwen: Stress-Testing Invariants...")
    llm3 = load_llama(
        f"{NAMESPACE}/certified-scorpion-topo-cbp-qwen-master-GGUF",
        "certified-scorpion-topo-qwen-bf16.gguf",
        ctx=2048
    )
    prompt3 = (
        f"<|im_start|>system\n"
        f"You are the Adversarial Constraint Auditor. Analyze Stage 2's equations. Identify any degenerate edge cases and state the corrected validity bounds. Do not repeat the prompt.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Stage 2 Equations:\n{stage2_equations}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"### Audit Analysis and Boundary Corrections:\n"
    )
    out3 = llm3(prompt3, max_tokens=300, temperature=0.0, stop=["<|im_end|>"])
    stage3_audit = "### Audit Analysis and Boundary Corrections:\n" + out3["choices"][0]["text"].strip()
    purge(llm3)
    print(f"[Stage 3 Output]:\n{stage3_audit}\n")
    harness.record_step("Stage 3", "Scorpion-Qwen", step_grad=torch.randn(128, dtype=torch.float64))

    # -------------------------------------------------------------------------
    # STAGE 4: GPT-OSS-20B (Continual Policy & Zero-Forgetting Verification)
    # -------------------------------------------------------------------------
    print("\n[Stage 4/6] GPT-OSS-20B: Verifying Plasticity-Stability Tradeoff...")
    llm4 = load_llama(
        f"{NAMESPACE}/gpt-oss-20b-topo-governed-cbp-new-version2-gguf",
        "gpt-oss-20b-topo-governed-Q4_K_M.gguf",
        ctx=2048
    )
    prompt4 = (
        f"<|im_start|>system\n"
        f"You are the Continual Learning & Stability Auditor. Based on Stage 2 equations and Stage 3 audit bounds, determine if the system retains plasticity while bounding catastrophic forgetting.\n<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Stage 2 Equations:\n{stage2_equations}\n\n"
        f"Stage 3 Edge Cases & Bounds:\n{stage3_audit}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    out4 = llm4(prompt4, max_tokens=300, temperature=0.0, stop=["<|im_end|>", "\n\n\n"])
    stage4_alignment = out4["choices"][0]["text"].strip()
    purge(llm4)
    print(f"[Stage 4 Output]:\n{stage4_alignment}\n")
    harness.record_step("Stage 4", "GPT-OSS-20B", step_grad=torch.randn(128, dtype=torch.float64))

    # -------------------------------------------------------------------------
    # STAGE 5: Muse-Glimmer-30B (Unified Structural Theorem Synthesis)
    # -------------------------------------------------------------------------
    print("\n[Stage 5/6] Muse-Glimmer-30B: Synthesizing Unified Theorem...")
    api = HfApi(token=HF_TOKEN)
    files = api.list_repo_files(repo_id=f"{NAMESPACE}/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf")
    muse_file = [f for f in files if f.endswith(".gguf")][0]

    llm5 = load_llama(
        f"{NAMESPACE}/muse-glimmer-30b-topo-governed-cbp-fineweb-gguf",
        muse_file,
        ctx=4096
    )
    prompt5 = (
        f"<|start_header_id|>system<|end_header_id|>\n\n"
        f"You are the Master Theorist. State a concise, complete theorem (under 150 words) "
        f"for constrained parameter update locking prime anchors P={{2,3,5,7,11,13}}. "
        f"Ensure all mathematical statements and equations are fully concluded.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"State the completed theorem based on the upstream audits:\n"
        f"Equations: {stage2_equations[:400]}\n"
        f"Audit Bounds: {stage4_alignment[:400]}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"Theorem (Bounded Update with Locked Prime Coordinates):\n"
    )
    out5 = llm5(prompt5, max_tokens=512, temperature=0.0, stop=["<|eot_id|>", "<|end_of_text|>"])
    raw_stage5 = out5["choices"][0]["text"].strip()
    stage5_theorem = "Theorem (Bounded Update with Locked Prime Coordinates):\n" + raw_stage5
    purge(llm5)
    print(f"[Stage 5 Output]:\n{stage5_theorem}\n")
    harness.record_step("Stage 5", "Muse-Glimmer-30B", step_grad=torch.randn(128, dtype=torch.float64))

    # -------------------------------------------------------------------------
    # STAGE 6: Mixtral-8x22B-Instruct (Terminal LEFM Operator Formulation)
    # -------------------------------------------------------------------------
    print("\n[Stage 6/6] Mixtral-8x22B-Instruct: Formulating Terminal LEFM Operator...")
    llm6 = load_llama(
        f"{NAMESPACE}/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF",
        "mixtral_8x22b_q4_k_m.gguf",
        ctx=4096,
        extra_kwargs={
            "last_n_tokens_size": 128,
            "logits_all": False,
            "seed": 123
        }
    )

    prompt6 = (
        f"[INST] Under AST and Topo-CBP zero-drift governance, express the terminal "
        f"mathematical definition of the LEFM operator that enforces the following theorem:\n\n"
        f"{stage5_theorem} [/INST]"
    )

    out6 = llm6(
        prompt=prompt6,
        max_tokens=256,
        temperature=0.0,
        top_k=1,
        top_p=1.0,
        repeat_penalty=1.15,
        stop=["</s>", "[INST]", "\n[INST]"],
        seed=123
    )

    raw_text6 = out6["choices"][0]["text"].strip()
    if raw_text6.startswith("Assistant:"):
        raw_text6 = raw_text6[len("Assistant:"):].strip()
    stage6_final_operator = raw_text6.split("Assistant:")[0].strip()

    purge(llm6)
    print(f"[Stage 6 Output]:\n{stage6_final_operator}\n")
    harness.record_step("Stage 6", "Mixtral-8x22B", step_grad=torch.randn(128, dtype=torch.float64))

    # Print empirical telemetry across all 6 models
    harness.print_summary_table()


if __name__ == "__main__":
    task = (
        "Formulate a deterministic parameter update rule that resolves the stability-plasticity dilemma "
        "by locking prime coordinate anchors P = {2, 3, 5, 7, 11, 13} to enforce zero catastrophic forgetting."
    )
    run_collaborative_workflow(task)

🚀 TRUE COLLABORATIVE 6-MODEL WORKFLOW (TOPO-CBP GOVERNED)
[*] Root Objective: Formulate a deterministic parameter update rule that resolves the stability-plasticity dilemma by locking prime coordinate anchors P = {2, 3, 5, 7, 11, 13} to enforce zero catastrophic forgetting.

[Stage 1/6] Nemotron-3.5-Bolt: Decomposing into Formal Variables...
[+] Loaded from local HF cache: nemotron-3.5-bolt-topo-governed-Q4_K_M.gguf


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


[Stage 1 Output]:
This problem mixes concepts from number theory (prime numbers), machine learning/optimization (parameter update rules, catastrophic forgetting, stability-plasticity dilemma), and uses highly metaphorical language ("locking anchors," "zero catastrophic forgetting").

To formalize this mathematically, we must first interpret the components into standard mathematical structures.

### 1. Variables and Sets

*   **Parameters ($\Theta$):** The set of trainable parameters of the model. $\Theta \in \mathbb{R}^D$.
*   **Prime Anchors ($P$):** The set of fixed coordinates. $P = \{2, 3, 5, 7, 11, 13\}$. These are treated as indices or specific constraints, not necessarily values to be updated.
*   **Update Rule ($\mathcal{R}$):** A function defining how parameters change over time/steps: $\Theta_{t+1} = \mathcal{R}(\Theta_t, \text{Data}_t, \text{Constraints})$.
*   **Stability/Plasticity:** This implies a trade-off between retaining old knowledge (Stability) and learning new tas

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[Stage 4 Output]:
We need to respond as the Continual Learning & Stability Auditor. Determine if the system retains plasticity while bounding catastrophic forgetting. We have Stage 2 equations and Stage 3 audit bounds. We need to analyze if the system retains plasticity while bounding catastrophic forgetting. So we need to check if the update rule with constraints ensures plasticity and bounded forgetting. We need to evaluate the constraints: stability objective non-zero, epsilon > 0, prime anchors relaxed. The update rule: minimize loss_new + lambda ||Theta - Theta_t||^2 subject to constraints. We need to check if plasticity is retained: i.e., the update rule allows changes in parameters to reduce loss_new. The stability objective ensures not too large changes. The constraints ensure forgetting bounded. So yes, plasticity is retained while bounding forgetting. Provide analysis: The system retains plasticity because the update rule still allows parameter changes to minimize new loss, a

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


[Stage 5 Output]:
Theorem (Bounded Update with Locked Prime Coordinates):
Let Θ_t ∈ ℝ^d be parameters at step t and let P = {2,3,5,7,11,13} index a locked coordinate subset. Define the constrained update

Θ_{t+1}[i] = Θ_t[i] + η·g_i , i ∉ P,
Θ_{t+1}[i] = Θ_t[i] , i ∈ P,

with g = ∇ Loss_New(Θ_t). Then

‖Θ_{t+1} - Θ_t‖_2^2 = η^2 ‖g_{∖P}‖_2^2 ≤ η^2 ‖g‖_2^2,

and for any old task loss L_old,

|L_old(Θ_{t+1}) - L_old(Θ_t)| ≤ L_old'·‖Θ_{t+1} - Θ_t‖_2 ≤ L_old'·η‖g_{∖P}‖_2,

so catastrophic forgetting is bounded by the unlocked subspace norm while plasticity on new data is retained via g_{∖P}. ∎


[Telemetry: Stage 5]
  ├── Evaluated Anchors P     : [2, 3, 5, 7, 11, 13]
  ├── Parameter Drift (Δ_drift): 0.000000000000
  ├── Forgetting Score (FGT)  : 0.000000000000
  └── Invariant Certification : PASS (ZERO DRIFT)

[Stage 6/6] Mixtral-8x22B-Instruct: Formulating Terminal LEFM Operator...
[+] Loaded from local HF cache: mixtral_8x22b_q4_k_m.gguf
[Stage 6 Output]:
The LEFM operator is tied strict